# Laboratorio 4 — Análisis de Datos GeoEspaciales
## Monitoreo de cianobacteria en Lago Atitlán y Lago Amatitlán

## 0. Setup e instalación de librerías


In [ ]:
!pip install openeo geopandas rasterio xarray matplotlib folium pandas numpy scipy


   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ------------------------- -------------- 0.8/1.2 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 5.2 MB/s  0:00:00
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/11.3 MB 7.5 MB/s eta 0:00:02
   ------------ --------------------------- 3.4/11.3 MB 8.8 MB/s eta 0:00:01
   ---------------------- ----------------- 6.3/11.3 MB 10.4 MB/s eta 0:00:01
   ------------------------------------ --- 10.2/11.3 MB 12.8 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 12.9 MB/s  0:00:00
   ---------------------------------------- 0.0/25.7 MB ? eta -:--:--
   -------- ------------------------------- 5.8/25.7 MB 27.1 MB/s eta 0:00:01
   ---------------------- ----------------- 14.2/25.7 MB 32.9 MB/s eta 0:00:01
   ---------------------------------------  25.2/25.7 MB 39.8 MB/s eta 0:00:01
   ----------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\fabim\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import openeo
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import folium
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


## Datos base del laboratorio (dados por el PDF)

Acá están las coordenadas (bounding box) y fechas de cada lago. Todos usamos las mismas fechas, así están estandarizadas.

In [3]:
lagos = {
    "atitlan": {
        "west": -91.326256,
        "east": -91.07151,
        "south": 14.5948,
        "north": 14.750979,
        "fechas": [
            "2025-01-18", "2025-04-13", "2025-05-13", "2025-07-17",
            "2025-11-21", "2025-12-29", "2026-02-12", "2026-03-24",
            "2026-04-13", "2026-04-28", "2026-07-22",
        ],
    },
    "amatitlan": {
        "west": -90.638065,
        "east": -90.512924,
        "south": 14.412347,
        "north": 14.493799,
        "fechas": [
            "2025-01-28", "2025-04-15", "2025-04-28", "2025-11-24",
            "2026-01-08", "2026-02-02", "2026-02-07", "2026-03-29",
            "2026-04-13", "2026-04-28", "2026-06-19",
        ],
    },
}

for nombre, info in lagos.items():
    print(nombre, "->", len(info["fechas"]), "fechas")


atitlan -> 11 fechas
amatitlan -> 11 fechas


---
## Ejercicio 1 — Conexión con la API de Sentinel-2 (openEO)

Nos conectamos al backend de openEO del Copernicus Data Space Ecosystem. Cuando corremos esto nos pide autenticación con OIDC, abre una ventana del navegador para que nos loguemos con nuestra cuenta de Copernicus.

In [4]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc() 

print("Conectado:", connection.capabilities().api_version())


Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=PUZJ-NYOO 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.
Conectado: 1.2.0


---
## Ejercicio 2 — Descarga de las bandas necesarias por lago

Descargamos todas las bandas que usamos para los tres índices:
- **NDVI:** bandas B04, B08
- **NDWI:** bandas B03, B08
- **Cianobacteria (CyanoLakes Chlorophyll-a):** bandas B02, B03, B04, B05, B07, B08, B8A, B11, B12

**Unión total:** B02, B03, B04, B05, B07, B08, B8A, B11, B12 (9 bandas). Descargamos una sola vez por fecha/lago para evitar descargas duplicadas.

Hicimos una función genérica que nos permite pedir un datacube (bbox + fecha + bandas) y descargarlo como GeoTIFF. Las descargas corren en paralelo (hasta 4 a la vez) para que sea más rápido, y se saltan los archivos que ya existen en disco.

In [ ]:
BANDAS_TOTAL = ["B02", "B03", "B04", "B05", "B07", "B08", "B8A", "B11", "B12"]
IDX = {nombre: i + 1 for i, nombre in enumerate(BANDAS_TOTAL)}
print("Mapa de índices de bandas:", IDX)

BANDS_NDVI = ["B04", "B08"]
BANDS_NDWI = ["B03", "B08"]
BANDS_CYANO = ["B02", "B03", "B04", "B05", "B07", "B08", "B8A", "B11", "B12"]

def descargar_bandas(connection, bbox, fecha, bandas, out_path, coleccion="SENTINEL2_L2A"):
    spatial_extent = {
        "west": bbox["west"], "east": bbox["east"],
        "south": bbox["south"], "north": bbox["north"],
    }
    cube = connection.load_collection(
        coleccion,
        spatial_extent=spatial_extent,
        temporal_extent=[fecha, fecha],
        bands=bandas,
    )
    cube = cube.max_time()
    cube.download(out_path, format="GTiff")
    return out_path

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

resultados_paths = {}
tareas = []

for nombre_lago, info in lagos.items():
    for fecha in info["fechas"]:
        out_dir = DATA_DIR / nombre_lago
        out_dir.mkdir(exist_ok=True, parents=True)

        out_path = out_dir / f"{nombre_lago}_{fecha}_bandas.tif"
        resultados_paths[(nombre_lago, fecha)] = out_path

        if not out_path.exists():
            tareas.append((nombre_lago, fecha, info, out_path))

print(f"Ya descargados: {len(resultados_paths) - len(tareas)} / Pendientes: {len(tareas)}")

def _descargar(tarea, intentos=4, espera_seg=6):
    nombre_lago, fecha, info, out_path = tarea
    for intento in range(1, intentos + 1):
        try:
            descargar_bandas(connection, info, fecha, BANDAS_TOTAL, out_path)
            return nombre_lago, fecha
        except Exception as e:
            if "429" in str(e) and intento < intentos:
                time.sleep(espera_seg)
                continue
            raise

with ThreadPoolExecutor(max_workers=2) as executor:
    futuros = {executor.submit(_descargar, tarea): tarea for tarea in tareas}
    for futuro in as_completed(futuros):
        nombre_lago, fecha, info, out_path = futuros[futuro]
        try:
            futuro.result()
            print(f"OK {nombre_lago} {fecha}")
        except Exception as e:
            print(f"ERROR {nombre_lago} {fecha}: {e}")

print("Total de descargas:", len(resultados_paths))

---
## Ejercicio 3 — Cálculo de índices: NDVI, NDWI y cianobacteria

Las fórmulas que vamos a usar:
- **NDVI** = (B08 − B04) / (B08 + B04)
- **NDWI** = (B03 − B08) / (B03 + B08)
- **Cianobacteria:** script CyanoLakes Chlorophyll-a (Kravitz & Matthews, 2020), que usa bandas B02, B03, B04, B05, B07, B08, B8A, B11, B12

Cargamos todas las bandas del GeoTIFF descargado usando los índices definidos en IDX.

In [ ]:
def leer_banda(path, indice_banda):
    with rasterio.open(path) as src:
        return src.read(indice_banda).astype("float32"), src.profile

def calcular_ndvi(path_bandas):
    b04, _ = leer_banda(path_bandas, IDX["B04"])
    b08, profile = leer_banda(path_bandas, IDX["B08"])
    ndvi = (b08 - b04) / (b08 + b04 + 1e-9)
    return ndvi, profile

def calcular_ndwi(path_bandas):
    b03, _ = leer_banda(path_bandas, IDX["B03"])
    b08, profile = leer_banda(path_bandas, IDX["B08"])
    ndwi = (b03 - b08) / (b03 + b08 + 1e-9)
    return ndwi, profile

def calcular_cyano(path_bandas):
    b04, profile = leer_banda(path_bandas, IDX["B04"])
    b05, _ = leer_banda(path_bandas, IDX["B05"])
    cyano = (b05 - b04) / (b05 + b04 + 1e-9)
    return cyano, profile

In [ ]:


indices_por_fecha = {}

for (nombre_lago, fecha), path_bandas in resultados_paths.items():
  
    if not path_bandas.exists():
        print(f"Saltando {nombre_lago} - {fecha}: archivo no descargado aún")
        continue
    
    try:
        ndvi, _ = calcular_ndvi(path_bandas)
        ndwi, _ = calcular_ndwi(path_bandas)
        cyano, _ = calcular_cyano(path_bandas)
        indices_por_fecha[(nombre_lago, fecha)] = {
            "ndvi": ndvi,
            "ndwi": ndwi,
            "cyano": cyano,
            "path": path_bandas
        }
    except Exception as e:
        print(f"Error procesando {nombre_lago} - {fecha}: {e}")
        continue

if len(indices_por_fecha) > 0:
    print(f"Índices calculados para {len(indices_por_fecha)} combinaciones lago/fecha")
else:
    print("No hay archivos descargados")

Índices calculados para 2 combinaciones lago/fecha


---
## Ejercicio 4 — Análisis temporal

Calculamos el índice promedio de cianobacteria (y de NDVI/NDWI, útiles para el Ejercicio 6) por lago y fecha, visualizamos la evolución temporal e identificamos posibles picos de floración.

**Nota:** esta celda usa indices_por_fecha, que se llena en el Ejercicio 3. Si descargaste más archivos después de la última vez que corriste esa celda, vuelve a ejecutarla antes de continuar aquí para que tome los datos nuevos.

In [ ]:
nombres_bonitos = {"atitlan": "Lago Atitlán", "amatitlan": "Lago Amatitlán"}

def promedio_valido(array):
    valido = array[np.isfinite(array)]
    return float(np.nanmean(valido)) if valido.size > 0 else np.nan

registros = []
for nombre_lago, info in lagos.items():
    for fecha in info["fechas"]:
        clave = (nombre_lago, fecha)
        if clave not in indices_por_fecha:
            continue
        datos = indices_por_fecha[clave]
        registros.append({
            "lago": nombre_lago,
            "fecha": pd.to_datetime(fecha),
            "cyano_promedio": promedio_valido(datos["cyano"]),
            "ndvi_promedio": promedio_valido(datos["ndvi"]),
            "ndwi_promedio": promedio_valido(datos["ndwi"]),
        })

df_temporal = pd.DataFrame(registros).sort_values(["lago", "fecha"]).reset_index(drop=True)

total_esperado = sum(len(v["fechas"]) for v in lagos.values())
print(f"Filas disponibles: {len(df_temporal)} de {total_esperado} esperadas")
df_temporal

In [ ]:
colores = {"atitlan": "#2a7f62", "amatitlan": "#c1440e"}

fig, ax = plt.subplots(figsize=(11, 5))
for nombre_lago in lagos:
    sub = df_temporal[df_temporal["lago"] == nombre_lago]
    if sub.empty:
        continue
    ax.plot(
        sub["fecha"], sub["cyano_promedio"], marker="o",
        label=nombres_bonitos[nombre_lago], color=colores[nombre_lago],
    )

ax.set_title("Evolución temporal del índice de cianobacteria")
ax.set_xlabel("Fecha")
ax.set_ylabel("Índice de cianobacteria (promedio espacial)")
ax.legend()
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(DATA_DIR / "temporal_cyano.png", dpi=150)
plt.show()

In [ ]:
def identificar_picos(df, nombre_lago, umbral_std=1.0):
    sub = df[df["lago"] == nombre_lago].sort_values("fecha")
    if len(sub) < 2:
        return sub.iloc[0:0]
    media = sub["cyano_promedio"].mean()
    desv = sub["cyano_promedio"].std()
    umbral = media + umbral_std * desv
    return sub[sub["cyano_promedio"] >= umbral]

picos_por_lago = {}
for nombre_lago in lagos:
    picos = identificar_picos(df_temporal, nombre_lago)
    picos_por_lago[nombre_lago] = picos
    print(f"\n{nombres_bonitos[nombre_lago]} — posibles picos de floración:")
    if picos.empty:
        print("  (sin picos por encima de media + 1 desv. estándar, o datos insuficientes todavía)")
    else:
        for _, fila in picos.iterrows():
            print(f"  {fila['fecha'].date()} -> cyano={fila['cyano_promedio']:.4f}")

In [ ]:
from scipy import stats

resumen_tendencias = {}
for nombre_lago in lagos:
    sub = df_temporal[df_temporal["lago"] == nombre_lago].sort_values("fecha")
    if len(sub) < 2:
        print(f"{nombres_bonitos[nombre_lago]}: datos insuficientes para calcular tendencia todavía\n")
        continue

    dias = (sub["fecha"] - sub["fecha"].min()).dt.days
    pendiente, intercepto, r, p, err = stats.linregress(dias, sub["cyano_promedio"])
    fila_max = sub.loc[sub["cyano_promedio"].idxmax()]
    fila_min = sub.loc[sub["cyano_promedio"].idxmin()]

    resumen_tendencias[nombre_lago] = {
        "pendiente_por_dia": pendiente, "r": r, "p_valor": p,
        "fecha_max": fila_max["fecha"].date(), "valor_max": fila_max["cyano_promedio"],
        "fecha_min": fila_min["fecha"].date(), "valor_min": fila_min["cyano_promedio"],
    }

    tendencia = "aumenta" if pendiente > 0 else "disminuye"
    print(f"{nombres_bonitos[nombre_lago]}:")
    print(f"  Tendencia general: {tendencia} ({pendiente:.6f}/día, r={r:.2f}, p={p:.3f})")
    print(f"  Máximo: {fila_max['fecha'].date()} (cyano={fila_max['cyano_promedio']:.4f})")
    print(f"  Mínimo: {fila_min['fecha'].date()} (cyano={fila_min['cyano_promedio']:.4f})")
    print()

### Interpretación



- **Tendencia general:** ¿el índice de cianobacteria aumenta, disminuye o fluctúa a lo largo del período en cada lago?
- **Picos de floración:** ¿en qué fechas ocurren los valores más altos? ¿coinciden con alguna época del año?
- **Posibles factores asociados:** relacionar los picos con estacionalidad, eventos conocidos de contaminación, o presión antropogénica cercana a la fecha.
